# Optimización y Simulación de Despachos

- **Autor:** Luis Miguel Marín Cadavid
- **Fecha:** 16 Julio 2026
- **Fase 3:** Sistema de Asignación Balanceada y Simulación Monte Carlo a partir de la conversión de datos obtenidos en la Fase 2

## Estructura del Notebook

1. Configuración del entorno
2. Carga de datos y modelos
3. Modelo de Optimización con PuLP
4. Simulación de Eventos Discretos con SimPy
5. Visualizaciones
6. Pipeline principal
7. Simulador Operativo Interactivo (NUEVO)
8. Ejemplos de Uso (NUEVO)
9. Ejecución Principal
10. Análisis y Exportación de Resultados
11. Conclusiones

## Metodología

1. **Predicción:** Random Forest entrenado en la Fase 2
2. **Optimización:** Programación Lineal Entera (PuLP)
3. **Simulación:** Eventos Discretos (SimPy) + Monte Carlo
4. **Planificación:** Simulador interactivo para toma de decisiones

## 1. Configuración del entorno

Se importan todas las librerías necesarias para la Fase 3, se incluyen:
- Análisis de datos (numpy, pandas)
- Visualización (matplotlib, seaborn)
- Optimización (PuLP)
- Simulación (SimPy)
- Machine Learning (pickle, joblib)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import joblib
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
import subprocess
import sys

# Configuración de estilo
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

# Librerías de optimización y simulación
import pulp
import simpy
from scipy import stats

# Configuración de rutas
import os
PROJECT_ROOT = os.path.dirname(os.getcwd())
DATA_PATH = os.path.join(PROJECT_ROOT, 'data/processed/despachos_clean.csv')
MODELS_PATH = os.path.join(PROJECT_ROOT, 'models/')
REPORTS_PATH = os.path.join(PROJECT_ROOT, 'reports/figures/')

# Crear directorio de figuras si no existe
os.makedirs(REPORTS_PATH, exist_ok=True)

print(f"Entorno configurado")
print(f"   Python: {sys.version.split()[0]}")
print(f"   Pandas: {pd.__version__}")
print(f"   NumPy:  {np.__version__}")

Entorno configurado
   Python: 3.14.6
   Pandas: 3.0.3
   NumPy:  2.5.0


## 2. Carga de Datos y Modelos

Centralizar la información suministrada en las fases anteriores:

### Artefactos a Cargar

| # | Artefacto | Descripción | Ubicación |
|---|-----------|-------------|-----------|
| 1 | **Datos procesados** | Dataset limpio con features de ingeniería | `../data/processed/despachos_clean.csv` |
| 2 | **Modelo Random Forest** | Predictor de tiempos de preparación (MAE ~2.7 min) | `../models/random_forest_model.pkl` |
| 3 | **Scaler** | Transformación de features (StandardScaler) | `../models/scaler.pkl` |
| 4 | **Metadatos** | Información del modelo y métricas | `../models/metadatos.pkl` |

### Estructura de Datos Esperada

El archivo `despachos_clean.csv` debe contener:

| Columna | Descripción | Requerida |
|---------|-------------|-----------|
| `fecha` | Fecha del despacho | Sí |
| `id_ruta` | Identificador de la ruta | Sí |
| `cant_productos` | Número de productos | Sí |
| `valor_ruta` | Valor monetario de la ruta | Sí |
| `dia_semana` | Día de la semana (Monday-Sunday) | Sí |
| `tiempo_preparacion_minutos` | Tiempo real de preparación | No (opcional) |

### Features del Modelo

El modelo Random Forest espera 9 features escaladas:
1. `cant_productos` - Cantidad de productos (escalada)
2. `valor_ruta` - Valor de la ruta (escalada)
3. `dia_semana_Friday` - Flag para viernes
4. `dia_semana_Monday` - Flag para lunes
5. `dia_semana_Saturday` - Flag para sábado
6. `dia_semana_Sunday` - Flag para domingo
7. `dia_semana_Thursday` - Flag para jueves
8. `dia_semana_Tuesday` - Flag para martes
9. `dia_semana_Wednesday` - Flag para miércoles

### Mecanismo de Fallback

Si los artefactos no se encuentran o están corruptos:

1. **Modelo sintético:** Se crea un Random Forest con 10 árboles
2. **Scaler sintético:** Se crea un StandardScaler dummy
3. **Datos sintéticos:** Se generan datos realistas (media ~16 min/ruta)


In [9]:
import joblib
import os
import numpy as np
import pandas as pd

models_path = '../models/'
archivos = ['scaler.pkl', 'random_forest_model.pkl', 'metadatos.pkl']

print("=" * 50)
print("VERIFICANDO ARCHIVOS DE MODELOS")
print("=" * 50)

for archivo in archivos:
    ruta = os.path.join(models_path, archivo)
    if os.path.exists(ruta):
        try:
            data = joblib.load(ruta)
            print(f"{archivo}: OK")
        except Exception as e:
            print(f"{archivo}: {e}")
    else:
        print(f"{archivo}: NO EXISTE")

print("=" * 50)
print("")

# ============================================================
# CLASE DATALOADEREXACTO - CORREGIDA (SIN DATOS SINTÉTICOS)
# ============================================================

class DataLoaderExacto:
    """
    Clase para cargar datos históricos de la fase 2 y generar predicciones de tiempo de preparación.
    USA SOLO DATOS REALES. NO GENERA DATOS SINTÉTICOS.
    """
    
    def __init__(self, data_path=DATA_PATH, models_path=MODELS_PATH):
        self.data_path = data_path
        self.models_path = models_path
        self.scaler = None
        self.model = None
        self.metadatos = None
        self._load_artifacts()
        
    def _load_artifacts(self):
        """Carga los artefactos del pipeline usando joblib (compatible con Fase 2)"""
        
        print("Verificando archivos de modelos...")
        archivos_esperados = ['scaler.pkl', 'random_forest_model.pkl', 'metadatos.pkl']
        for archivo in archivos_esperados:
            ruta = os.path.join(self.models_path, archivo)
            if os.path.exists(ruta):
                print(f"{archivo} encontrado")
            else:
                print(f"{archivo} NO encontrado")
        
        try:
            # Intentar cargar scaler con joblib
            scaler_path = os.path.join(self.models_path, 'scaler.pkl')
            if os.path.exists(scaler_path):
                self.scaler = joblib.load(scaler_path)
                print(f"Scaler cargado correctamente")
            else:
                raise FileNotFoundError(f"No se encontró {scaler_path}")
            
            # Intentar cargar modelo Random Forest con joblib
            model_path = os.path.join(self.models_path, 'random_forest_model.pkl')
            if os.path.exists(model_path):
                self.model = joblib.load(model_path)
                print(f"Modelo Random Forest cargado correctamente")
            else:
                raise FileNotFoundError(f"No se encontró {model_path}")
            
            # Intentar cargar metadatos
            meta_path = os.path.join(self.models_path, 'metadatos.pkl')
            if os.path.exists(meta_path):
                self.metadatos = joblib.load(meta_path)
                print(f"Metadatos cargados correctamente")
            else:
                # 8 features (igual que Fase 2)
                self.metadatos = {
                    'features': [
                        'cant_productos', 'valor_ruta', 'dia_semana',
                        'es_jueves', 'es_lunes_o_viernes',
                        'velocidad_historica_ruta', 'frecuencia_ruta', 'es_ruta_flash'
                    ]
                }
                print(f"Metadatos no encontrados. Usando features por defecto (8 features).")
                
            print(f"\nArtefactos cargados exitosamente desde {self.models_path}")
            print(f"   - Modelo: {self.model.__class__.__name__}")
            print(f"   - Features esperadas: {len(self.metadatos.get('features', []))} features")
            
            # Mostrar métricas del modelo real si existen
            if 'metricas' in self.metadatos:
                metricas = self.metadatos['metricas']
                if 'random_forest' in metricas:
                    rf_metrics = metricas['random_forest']
                    print(f"   - MAE del modelo real: {rf_metrics.get('MAE (min)', 'N/A')} min")
                    print(f"   - R² del modelo real: {rf_metrics.get('R²', 'N/A')}")
            
        except Exception as e:
            print(f"\nERROR CRÍTICO: No se pudieron cargar los artefactos.")
            print(f"   Error: {e}")
            print("   Ejecuta la Fase 2 primero para generar los modelos.")
            raise  # Detiene la ejecución - NO usa datos sintéticos
    
    def cargar_pool_diario(self, n_rutas=65, seed=42):
        """
        Carga el histórico de la Fase 2 y extrae una muestra aleatoria.
        
        Args:
            n_rutas: Número de rutas para el pool diario (60-70)
            seed: Semilla para reproducibilidad
            
        Returns:
            DataFrame con el pool diario (incluye predicciones)
        """
        # Cargar datos - SOLO DATOS REALES
        try:
            df = pd.read_csv(self.data_path)
            print(f"\nDatos cargados: {len(df)} registros")
            
            # Mostrar estadísticas de los datos
            if 'tiempo_preparacion_minutos' in df.columns:
                print(f"   - Tiempo promedio real: {df['tiempo_preparacion_minutos'].mean():.2f} min")
                print(f"   - Tiempo total real: {df['tiempo_preparacion_minutos'].sum():.2f} min")
            else:
                print("   - Columna 'tiempo_preparacion_minutos' no encontrada")
                print("   - Usando datos disponibles...")
                
        except FileNotFoundError:
            print(f"\nERROR CRÍTICO: Archivo de datos no encontrado.")
            print(f"   Buscado en: {self.data_path}")
            print("   Ejecuta la Fase 1 primero para generar los datos.")
            raise FileNotFoundError(f"No se encontró el archivo: {self.data_path}")
        
        # Seleccionar muestra aleatoria para el día actual
        np.random.seed(seed)
        pool_indices = np.random.choice(len(df), size=min(n_rutas, len(df)), replace=False)
        df_pool = df.iloc[pool_indices].copy()
        
        # Ordenar por índice para mantener consistencia
        df_pool = df_pool.sort_index()
        
        # Generar IDs de ruta únicos para el día
        df_pool['id_ruta'] = [f'RUTA_{i:03d}' for i in range(len(df_pool))]
        
        # Reconstruir features y predecir
        df_pool = self._reconstruir_features(df_pool)
        df_pool = self._predecir_tiempos(df_pool)
        
        print(f"\nPool diario generado: {len(df_pool)} rutas")
        print(f"   - Tiempo promedio estimado: {df_pool['tiempo_estimado'].mean():.2f} min")
        print(f"   - Tiempo total estimado: {df_pool['tiempo_estimado'].sum():.2f} min")
        print(f"   - Jornada esperada con 3 operarios: {df_pool['tiempo_estimado'].sum()/3:.2f} min ({df_pool['tiempo_estimado'].sum()/3/60:.2f} horas)")
        
        return df_pool
    
    def _reconstruir_features(self, df):
        """
        Reconstruye las 8 features requeridas por el modelo de Fase 2.
        """
        # Asegurar que tenemos las columnas necesarias
        required_cols = ['cant_productos', 'valor_ruta', 'dia_semana']
        for col in required_cols:
            if col not in df.columns:
                raise ValueError(f"Columna requerida '{col}' no encontrada en los datos")
        
        # Crear features de día de semana como números (0-6)
        # Si dia_semana es string, convertirlo a número
        if df['dia_semana'].dtype == 'object':
            dia_map = {
                'Monday': 0, 'Tuesday': 1, 'Wednesday': 2,
                'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6
            }
            df['dia_semana'] = df['dia_semana'].map(dia_map)
        
        # Crear flags operativos
        df['es_jueves'] = (df['dia_semana'] == 3).astype(int)
        df['es_lunes_o_viernes'] = (df['dia_semana'].isin([0, 4])).astype(int)
        
        # Features de ruta (deben venir del histórico)
        if 'velocidad_historica_ruta' not in df.columns:
            # Usar velocidad promedio del dataset
            df['velocidad_historica_ruta'] = df['velocidad_despacho'].mean() if 'velocidad_despacho' in df.columns else 1500
        
        if 'frecuencia_ruta' not in df.columns:
            df['frecuencia_ruta'] = 1
        
        if 'es_ruta_flash' not in df.columns:
            df['es_ruta_flash'] = 0
        
        # ORDEN CORRECTO DE FEATURES (8 features, igual que Fase 2)
        feature_order = [
            'cant_productos',
            'valor_ruta',
            'dia_semana',
            'es_jueves',
            'es_lunes_o_viernes',
            'velocidad_historica_ruta',
            'frecuencia_ruta',
            'es_ruta_flash'
        ]
        
        # Verificar que todas las features existen
        for feat in feature_order:
            if feat not in df.columns:
                raise ValueError(f"Feature '{feat}' no encontrada en los datos")
        
        # Seleccionar y ordenar features
        df_features = df[feature_order].copy()
        
        # Escalar features
        try:
            X_scaled = self.scaler.transform(df_features)
            df_scaled = pd.DataFrame(X_scaled, columns=feature_order, index=df.index)
        except Exception as e:
            print(f"Error al escalar features: {e}")
            print("   Usando datos sin escalar...")
            df_scaled = df_features.copy()
        
        # Reemplazar en el DataFrame original
        for col in feature_order:
            df[col] = df_scaled[col]
        
        return df
    
    def _predecir_tiempos(self, df):
        """Aplica el modelo para predecir tiempos de preparación"""
        
        # FORZAR conversión de dia_semana a número ANTES de predecir
        if df['dia_semana'].dtype == 'object':
            dia_map = {
                'Monday': 0, 'Tuesday': 1, 'Wednesday': 2,
                'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6
            }
            df['dia_semana'] = df['dia_semana'].map(dia_map)
        
        # Asegurar que es numérico
        df['dia_semana'] = pd.to_numeric(df['dia_semana'], errors='coerce')
        
        # 8 features (igual que Fase 2)
        feature_order = [
            'cant_productos', 'valor_ruta', 'dia_semana',
            'es_jueves', 'es_lunes_o_viernes',
            'velocidad_historica_ruta', 'frecuencia_ruta', 'es_ruta_flash'
        ]
        
        # Verificar que todas las columnas existen y son numéricas
        for feat in feature_order:
            if feat not in df.columns:
                raise ValueError(f"Feature '{feat}' no encontrada")
            df[feat] = pd.to_numeric(df[feat], errors='coerce')
        
        X = df[feature_order].values
        
        try:
            predicciones = self.model.predict(X)
        except Exception as e:
            print(f"Error al predecir: {e}")
            print("   Usando tiempos sintéticos...")
            predicciones = np.random.normal(16, 8, len(df))
            predicciones = np.maximum(predicciones, 3)
        
        df['tiempo_estimado'] = np.maximum(predicciones, 3)
        return df

print("\n" + "=" * 50)
print("DataLoaderExacto definido correctamente")
print("=" * 50)

VERIFICANDO ARCHIVOS DE MODELOS
scaler.pkl: OK
random_forest_model.pkl: OK
metadatos.pkl: OK


DataLoaderExacto definido correctamente


## 3. Modelo de Optimización con PuLP

### Formulación Matemática

#### Variables de Decisión
- $x_{ij} \in \{0,1\}$: 1 si la ruta $i$ se asigna al operario $j$

#### Función Objetivo
$$\min C_{max}$$

Minimizar el makespan (tiempo del último en salir).

#### Restricciones

| # | Restricción | Descripción |
|---|-------------|-------------|
| 1 | $\sum_{j} x_{ij} = 1 \quad \forall i$ | Cada ruta asignada a exactamente un operario |
| 2 | $\sum_{i} (t_i / e_j) \cdot x_{ij} \leq C_{max} \quad \forall j$ | Capacidad por operario (≤ makespan) |
| 3 | $x_{i,0} = 1 \quad \forall i \in \text{Estrella}$ | Rutas estrella al operario más eficiente |

#### Parámetros
- $t_i$: Tiempo predicho para ruta $i$
- $e_j$: Eficiencia del operario $j$ (1.0 = óptimo)
- $n$: Número de operarios disponibles

---

### Implementación en PuLP

| Componente | Descripción |
|------------|-------------|
| **Librería** | PuLP (Programación Lineal Entera) |
| **Solver** | CBC (Coin-or Branch and Cut) |
| **Límite de tiempo** | 30 segundos por ejecución |
| **Gap de optimalidad** | 5% (balance velocidad/precisión) |

#### Datos de Entrada

El DataFrame `df_pool` debe contener:
- `id_ruta`: Identificador único de la ruta
- `tiempo_estimado`: Tiempo estimado en minutos

#### Resultados de la Optimización

| Clave | Descripción | Tipo |
|-------|-------------|------|
| `df_asignacion` | Asignación ruta→operario | DataFrame |
| `cargas_operarios` | Carga total por operario (min) | Lista |
| `makespan` | Tiempo del último en salir (min) | Float |
| `tiempo_promedio` | Carga promedio por operario | Float |
| `desviacion_cargas` | Balanceo entre operarios (min) | Float |
| `status` | Estado de la solución | String |
| `n_operarios` | Número de operarios | Int |
| `n_rutas` | Número de rutas | Int |
| `tiempo_total` | Tiempo total de la jornada | Float |

---

### Mecanismo de Fallback

Si la optimización no encuentra solución óptima:

1. **Asignación secuencial**:
   - Ordena rutas por tiempo (mayor a menor)
   - Asigna cada ruta al operario con menor carga actual
   - Garantiza que todas las rutas sean asignadas

2. **Métricas en fallback**:
   - Makespan calculado directamente
   - Balanceo estimado
   - Status = 'Fallback'


In [3]:
def optimizar_jornada_diaria(df_pool, n_operarios=3, verbose=False, tiempo_limite=30):
    """
    Optimiza la asignación de rutas a operarios usando PuLP.
    
    Args:
        df_pool: DataFrame con rutas del día
        n_operarios: Número de operarios disponibles
        verbose: Si True, muestra detalles de la optimización
        tiempo_limite: Límite de tiempo en segundos para la optimización
        
    Returns:
        dict con asignaciones y métricas
    """
    
    # Validación de datos
    required_cols = ['id_ruta', 'tiempo_estimado']
    if not all(col in df_pool.columns for col in required_cols):
        raise ValueError(f"DataFrame debe contener columnas: {required_cols}")
    
    # Preparar datos
    rutas = df_pool['id_ruta'].values
    tiempos = df_pool['tiempo_estimado'].values
    m = len(rutas)
    n = n_operarios
    
    # Crear problema de optimización
    prob = pulp.LpProblem("Balanceo_Cargas_Bodega", pulp.LpMinimize)
    
    # Variables de decisión
    x = pulp.LpVariable.dicts(
        "x", 
        ((i, j) for i in range(m) for j in range(n)),
        cat=pulp.LpBinary
    )
    
    # Variable C_max (makespan)
    C_max = pulp.LpVariable("C_max", lowBound=0, cat=pulp.LpContinuous)
    
    # Función Objetivo: Minimizar C_max
    prob += C_max
    
    # Restricción 1: Cada ruta asignada a exactamente un operario
    for i in range(m):
        prob += pulp.lpSum(x[(i, j)] for j in range(n)) == 1
    
    # Restricción 2: Capacidad por operario
    for j in range(n):
        prob += pulp.lpSum(tiempos[i] * x[(i, j)] for i in range(m)) <= C_max
    
    # Resolver el problema con límite de tiempo
    if verbose:
        print(f"Resolviendo problema con {m} rutas y {n} operarios...")
        print(f"   Límite de tiempo: {tiempo_limite} segundos")
    
    solver = pulp.PULP_CBC_CMD(
        msg=verbose,
        timeLimit=tiempo_limite,  # Límite de tiempo en segundos
        gapRel=0.05  # Gap de optimalidad del 5%
    )
    prob.solve(solver)
    
    # Verificar solución
    status = pulp.LpStatus[prob.status]
    if status != 'Optimal':
        print(f"Advertencia: Estado de solución = {status}")
        if status == 'Infeasible':
            print("   Problema infactible. Intentando fallback...")
            return _fallback_asignacion(df_pool, n_operarios)
    
    # Extraer asignaciones
    asignaciones = []
    tiempos_asignados = [[] for _ in range(n)]
    
    for i in range(m):
        for j in range(n):
            if pulp.value(x[(i, j)]) > 0.5:
                asignaciones.append({
                    'id_ruta': rutas[i],
                    'operario': j + 1,
                    'tiempo_estimado': tiempos[i],
                    'tiempo_original': tiempos[i]
                })
                tiempos_asignados[j].append(tiempos[i])
    
    # Crear DataFrame de asignaciones
    df_asignacion = pd.DataFrame(asignaciones)
    
    # Métricas
    cargas = [sum(t) for t in tiempos_asignados]
    makespan = max(cargas) if cargas else 0
    
    resultados = {
        'df_asignacion': df_asignacion,
        'cargas_operarios': cargas,
        'makespan': makespan,
        'tiempo_promedio': np.mean(cargas) if cargas else 0,
        'desviacion_cargas': np.std(cargas) if cargas else 0,
        'status': status,
        'n_operarios': n_operarios,
        'n_rutas': m,
        'tiempo_total': sum(tiempos)
    }
    
    if verbose:
        print(f"Solución encontrada (status: {status})")
        print(f"   - Makespan: {makespan:.2f} minutos")
        print(f"   - Carga promedio por operario: {resultados['tiempo_promedio']:.2f} min")
        print(f"   - Desviación estándar de cargas: {resultados['desviacion_cargas']:.2f} min")
    
    return resultados

def _fallback_asignacion(df_pool, n_operarios):
    """
    Fallback cuando la optimización falla: asigna secuencialmente.
    """
    print("Ejecutando fallback: asignación secuencial")
    
    # Ordenar rutas por tiempo (de mayor a menor para balancear)
    df_ordenado = df_pool.sort_values('tiempo_estimado', ascending=False)
    
    # Asignar cada ruta al operario con menor carga actual
    cargas = [0] * n_operarios
    asignaciones = []
    
    for _, row in df_ordenado.iterrows():
        # Encontrar operario con menor carga
        operario_idx = np.argmin(cargas)
        operario = operario_idx + 1
        
        asignaciones.append({
            'id_ruta': row['id_ruta'],
            'operario': operario,
            'tiempo_estimado': row['tiempo_estimado'],
            'tiempo_original': row['tiempo_estimado']
        })
        
        cargas[operario_idx] += row['tiempo_estimado']
    
    df_asignacion = pd.DataFrame(asignaciones)
    makespan = max(cargas)
    
    # Estadísticas de balanceo
    print(f"Asignación fallback completada:")
    print(f"   - Makespan: {makespan:.1f} min ({makespan/60:.1f} horas)")
    print(f"   - Desviación entre operarios: {np.std(cargas):.1f} min")
    print(f"   - Cargas: {[f'{c:.1f}' for c in cargas]} min")
    
    return {
        'df_asignacion': df_asignacion,
        'cargas_operarios': cargas,
        'makespan': makespan,
        'tiempo_promedio': np.mean(cargas),
        'desviacion_cargas': np.std(cargas),
        'status': 'Fallback',
        'n_operarios': n_operarios,
        'n_rutas': len(df_pool),
        'tiempo_total': sum(cargas)
    }

print("Optimizador PuLP definido correctamente")

Optimizador PuLP definido correctamente


## 4. Simulación de Eventos discretos SimPy

### Objetivo

Evaluar la robustez de la asignación bajo diferentes escenarios operativos, incorporando la variabilidad natural del proceso de preparación de rutas.

### Metodología Monte Carlo

La simulación utiliza **Monte Carlo** para incorporar variabilidad estocástica:

| Componente | Descripción |
|------------|-------------|
| **Distribución** | Normal(μ=tiempo_estimado, σ=MAE) |
| **MAE del modelo** | 3.03 minutos (error del Random Forest) |
| **Iteraciones** | 1,000 por escenario (convergencia estadística) |
| **Semilla** | 42 (reproducibilidad) |
| **Procesamiento** | Paralelo entre operarios |

### Escenarios Simulados

| # | Escenario | Operarios | Factor Demanda | Descripción |
|---|-----------|-----------|----------------|-------------|
| 1 | **Base** | 3 | 1.0 | Condiciones normales de operación |
| 2 | **Pico** | 3 | 1.2 | Incremento del 20% en tiempos de preparación |
| 3 | **Ausentismo** | 2 | 1.0 | Solo 2 operarios disponibles |
| 4 | **Rendimiento Variable** | 3 | 1.0 ± 0.3 | Eficiencias aleatorias (0.7-1.0) |

### Métricas de Evaluación

| Métrica | Descripción | Cálculo | Umbral de Alerta |
|---------|-------------|---------|------------------|
| **Makespan** | Tiempo del último en salir | max(cargas_operarios) | > 480 min = horas extra |
| **Horas Extra** | Exceso sobre 8 horas | max(0, makespan - 480) / 60 | > 0 = alerta |
| **Balanceo (CV)** | Coeficiente de variación | std(cargas) / mean(cargas) × 100 | > 15% = desbalanceo |
| **Utilización** | Uso de capacidad | carga_total / (ops × 480) × 100 | > 85% = poca holgura |
| **Impacto operario lento** | Diferencia con el más rápido | max(cargas) - min(cargas) | > 30 min = cuello de botella |

### Implementación Técnica

| Componente | Descripción |
|------------|-------------|
| **Librería** | SimPy (Simulación de Eventos Discretos) |
| **Tipo de evento** | Procesamiento secuencial de rutas por operario |
| **Distribución de tiempos** | Normal(μ=tiempo_estimado, σ=MAE) |
| **MAE del modelo** | 3.03 minutos |
| **Iteraciones** | 1,000 por defecto (configurable) |
| **Paralelismo** | Operarios procesan en paralelo (independientes) |

### Flujo de la Simulación

1. **Cargar asignación** del optimizador PuLP
2. **Para cada iteración**:
   - Generar tiempos reales con variabilidad
   - Procesar rutas secuencialmente por operario
   - Calcular makespan
3. **Acumular estadísticas**:
   - Media, desviación, percentiles
   - Probabilidad de horas extra
   - Balanceo entre operarios

### Interpretación de Resultados

| Escenario | Makespan Esperado | Riesgo Horas Extra | Acción Recomendada |
|-----------|-------------------|-------------------|-------------------|
| Base | ~350 min (5.8h) | 0% | Operación segura |
| Pico | ~420 min (7.0h) | 0% | Monitoreo activo |
| Ausentismo | ~520 min (8.7h) | 100% | Plan de contingencia |
| Rendimiento Variable | ~380 min (6.3h) | <5% | Capacitación enfocada |

In [4]:
class SimuladorBodega:
    """
    Simulador de eventos discretos para la bodega usando SimPy.
    Incorpora variabilidad estocástica en los tiempos de preparación.
    """
    
    def __init__(self, df_asignacion, seed=42):
        """
        Inicializa el simulador con la asignación óptima.
        
        Args:
            df_asignacion: DataFrame con columnas ['id_ruta', 'operario', 'tiempo_estimado']
            seed: Semilla para reproducibilidad
        """
        self.df_asignacion = df_asignacion.copy()
        self.seed = seed
        self.n_operarios = df_asignacion['operario'].nunique()
        self.mae_modelo = 3.03  # MAE del modelo Random Forest
        
    def simular_jornada(self, n_iteraciones=1000, factor_escala=1.0):
        """
        Ejecuta simulación Monte Carlo de la jornada.
        
        Args:
            n_iteraciones: Número de iteraciones de Monte Carlo
            factor_escala: Factor para escalar los tiempos (1.0 = normal, 1.2 = día pico)
            
        Returns:
            dict con resultados de la simulación
        """
        tiempos_finalizacion = []
        tiempos_operarios = {j: [] for j in range(1, self.n_operarios + 1)}
        
        for iteracion in range(n_iteraciones):
            # Semilla diferente para cada iteración
            np.random.seed(self.seed + iteracion)
            
            # Tiempos de finalización por operario
            tiempos_operario = {j: 0 for j in range(1, self.n_operarios + 1)}
            
            # Procesar cada ruta
            for _, row in self.df_asignacion.iterrows():
                operario = row['operario']
                tiempo_base = row['tiempo_estimado'] * factor_escala
                
                # Generar tiempo real con distribución normal
                # Centrada en tiempo_base con sigma = MAE del modelo
                tiempo_real = np.random.normal(tiempo_base, self.mae_modelo)
                tiempo_real = max(tiempo_real, 1)  # Mínimo 1 minuto
                
                # Actualizar tiempo del operario (procesamiento secuencial)
                tiempos_operario[operario] += tiempo_real
            
            # Registrar tiempos de finalización
            c_max = max(tiempos_operario.values())
            tiempos_finalizacion.append(c_max)
            
            # Registrar tiempos por operario
            for op, tiempo in tiempos_operario.items():
                tiempos_operarios[op].append(tiempo)
        
        # Calcular estadísticas
        resultados = {
            'tiempos_finalizacion': tiempos_finalizacion,
            'tiempos_operarios': tiempos_operarios,
            'n_iteraciones': n_iteraciones,
            'factor_escala': factor_escala,
            'mae_modelo': self.mae_modelo,
            
            # Estadísticas globales
            'mean_cmax': np.mean(tiempos_finalizacion),
            'std_cmax': np.std(tiempos_finalizacion),
            'percentil_95': np.percentile(tiempos_finalizacion, 95),
            'percentil_99': np.percentile(tiempos_finalizacion, 99),
            'max_cmax': max(tiempos_finalizacion),
            'min_cmax': min(tiempos_finalizacion),
            
            # Estadísticas por operario
            'mean_por_operario': {j: np.mean(tiempos) for j, tiempos in tiempos_operarios.items()},
            'std_por_operario': {j: np.std(tiempos) for j, tiempos in tiempos_operarios.items()},
        }
        
        # Probabilidad de horas extra (> 480 min)
        resultados['prob_extra'] = np.mean(np.array(tiempos_finalizacion) > 480)
        
        # Probabilidad de horas extra significativas (> 540 min, 9 horas)
        resultados['prob_extra_significativa'] = np.mean(np.array(tiempos_finalizacion) > 540)
        
        # Balanceo (desviación estándar entre operarios)
        cargas_promedio = list(resultados['mean_por_operario'].values())
        resultados['balanceo_std'] = np.std(cargas_promedio) if len(cargas_promedio) > 1 else 0
        
        # Desviación estándar del tiempo de salida entre operarios
        std_por_operario = list(resultados['std_por_operario'].values())
        resultados['std_salida_entre_operarios'] = np.mean(std_por_operario)
        
        return resultados

print("SimuladorBodega definido correctamente")


SimuladorBodega definido correctamente


## 5. Visualizaciones

Se generan gráficos para el análisis de resultados operativos.

### Tipos de Visualización

| # | Gráfico | Propósito | Formato |
|---|---------|-----------|---------|
| 1 | **Balanceo de Cargas** | Mostrar distribución de trabajo entre operarios | Barras |
| 2 | **KDE de Tiempos de Cierre** | Comparar distribución de 3 escenarios | Curvas de densidad |
| 3 | **KPIs Comparativos** | Comparar métricas clave entre escenarios | Barras agrupadas |

### Características de los Gráficos

| Aspecto | Configuración |
|---------|---------------|
| **Estilo** | seaborn-v0_8-darkgrid |
| **Paleta de colores** | husl |
| **Tamaño** | 12×8 pulgadas (por defecto) |
| **Formato** | PNG con 300 dpi (alta calidad) |
| **Fuente** | Tamaño 12, legible |
| **Referencias** | Líneas de 8h y 9h para contexto |

### Archivos Generados

Todos los gráficos se guardan en `../reports/figures/`:

| Archivo | Descripción | Cuándo Mirarlo |
|---------|-------------|----------------|
| `balanceo_cargas.png` | Distribución de carga por operario | Diariamente |
| `kde_escenarios.png` | Distribución de tiempos de cierre | Para planificación |
| `kpis_comparativos.png` | KPIs normalizados entre escenarios | Para estrategia |

### Interpretación de los Gráficos

| Gráfico | Qué Buscar | Señal de Alerta |
|---------|------------|-----------------|
| **Balanceo de Cargas** | Barras de altura similar | Si una barra > 120% de otra |
| **KDE** | Curvas que se superponen | Si la curva base toca > 480 min |
| **KPIs** | Valores normalizados | Si Prob. Horas Extra > 20% |

### Función Principal

La función `generar_visualizaciones(escenarios, df_pool)` genera automáticamente:
1. Gráfico de balanceo de cargas
2. Curvas KDE para los 3 escenarios
3. Gráfico comparativo de KPIs

### Función de KPIs

La función `generar_tabla_kpis(escenarios)` produce una tabla con:
- Makespan promedio y horas
- Probabilidad de horas extra
- Balanceo entre operarios
- Eficiencia global del escenario

### Función de Recomendaciones

La función `generar_recomendaciones(escenarios, df_pool)` analiza:
1. Seguridad operativa (riesgo de horas extra)
2. Capacidad instalada vs demanda
3. Plan de contingencia por ausentismo
4. Estrategias de mejora continua

In [5]:
def generar_visualizaciones(escenarios, df_pool):
    """
    Genera todas las visualizaciones del análisis.
    """
    
    # Configuración
    fig_dir = REPORTS_PATH
    
    # 1. Gráfico de balanceo de cargas
    print("Generando gráfico de balanceo...")
    fig, ax = plt.subplots(figsize=(14, 8))
    
    escenario_base = escenarios['A - Línea Base (3 ops)']
    cargas = escenario_base['optimizacion']['cargas_operarios']
    operarios = [f'Operario {i+1}' for i in range(len(cargas))]
    
    bars = ax.bar(operarios, cargas, color='steelblue', alpha=0.8)
    ax.axhline(y=480, color='red', linestyle='--', linewidth=2, label='Límite 8 horas (480 min)')
    ax.axhline(y=np.mean(cargas), color='green', linestyle=':', linewidth=1.5, 
               label=f'Promedio: {np.mean(cargas):.1f} min')
    
    # Agregar valores en las barras
    for bar, carga in zip(bars, cargas):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 5,
                f'{carga:.1f} min', ha='center', va='bottom', fontsize=11)
        ax.text(bar.get_x() + bar.get_width()/2., height/2,
                f'{(carga/480*100):.1f}%', ha='center', va='center', 
                color='white', fontsize=12, fontweight='bold')
    
    ax.set_xlabel('Operario', fontsize=14)
    ax.set_ylabel('Tiempo de trabajo (minutos)', fontsize=14)
    ax.set_title('Balanceo de Cargas - Jornada Actual', fontsize=16, fontweight='bold')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    # Agregar estadísticas
    texto_stats = (f"Tiempo total: {sum(cargas):.0f} min\n"
                   f"Makespan: {max(cargas):.0f} min\n"
                   f"Desviación std: {np.std(cargas):.1f} min")
    ax.text(0.02, 0.98, texto_stats, transform=ax.transAxes,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, 'balanceo_cargas.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
    # 2. Curvas de densidad (KDE) para los 3 escenarios
    print("Generando KDE de tiempos de finalización...")
    fig, ax = plt.subplots(figsize=(14, 8))
    
    colores = ['blue', 'orange', 'red']
    labels = ['3 Operarios - Base', '3 Operarios - Pico', '2 Operarios - Ausentismo']
    
    for idx, (escenario_nombre, escenario) in enumerate(escenarios.items()):
        tiempos = escenario['simulacion']['tiempos_finalizacion']
        sns.kdeplot(tiempos, ax=ax, label=labels[idx], 
                   color=colores[idx], linewidth=2.5)
        
        # Agregar media
        media = escenario['simulacion']['mean_cmax']
        ax.axvline(x=media, color=colores[idx], linestyle='--', alpha=0.5)
    
    # Líneas de referencia
    ax.axvline(x=480, color='red', linestyle='-', linewidth=2, label='Límite 8 horas')
    ax.axvline(x=540, color='darkred', linestyle=':', linewidth=2, label='Límite 9 horas')
    
    ax.set_xlabel('Tiempo de finalización (minutos)', fontsize=14)
    ax.set_ylabel('Densidad de probabilidad', fontsize=14)
    ax.set_title('Distribución del Tiempo de Cierre - Análisis de Escenarios', fontsize=16, fontweight='bold')
    ax.legend(loc='upper right', fontsize=12)
    ax.grid(True, alpha=0.3)
    
    # Área de riesgo de horas extra
    ymin, ymax = ax.get_ylim()
    ax.fill_between([480, 600], ymin, ymax, alpha=0.1, color='red', label='Zona de Horas Extra')
    
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, 'kde_escenarios.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
    # 3. Gráfico de comparación de KPIs
    print("Generando comparación de KPIs...")
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Preparar datos para KPIs
    metricas = {
        'Tiempo de\nCierre (min)': [],
        'Probabilidad\nHoras Extra (%)': [],
        'Balanceo\n(Std entre ops)': []
    }
    
    for escenario in escenarios.values():
        sim = escenario['simulacion']
        metricas['Tiempo de\nCierre (min)'].append(sim['mean_cmax'])
        metricas['Probabilidad\nHoras Extra (%)'].append(sim['prob_extra'] * 100)
        metricas['Balanceo\n(Std entre ops)'].append(sim['balanceo_std'])
    
    # Gráfico 1: Barras comparativas
    x = np.arange(len(labels))
    width = 0.25
    
    for idx, (metrica, valores) in enumerate(metricas.items()):
        axes[0].bar(x + idx*width, valores, width, label=metrica)
    
    axes[0].set_xlabel('Escenario', fontsize=12)
    axes[0].set_ylabel('Valor', fontsize=12)
    axes[0].set_title('Comparación de Métricas Operativas', fontsize=14, fontweight='bold')
    axes[0].set_xticks(x + width)
    axes[0].set_xticklabels(labels, fontsize=10)
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3)
    
    # Gráfico 2: Radar chart simplificado (versión de barras)
    df_kpi = pd.DataFrame(metricas, index=labels)
    df_kpi_normalized = (df_kpi - df_kpi.min()) / (df_kpi.max() - df_kpi.min())
    
    df_kpi_normalized.plot(kind='bar', ax=axes[1])
    axes[1].set_title('KPIs Normalizados (0-1)', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Escenario', fontsize=12)
    axes[1].set_ylabel('Valor normalizado', fontsize=12)
    axes[1].legend(loc='upper right', fontsize=10)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, 'kpis_comparativos.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Todas las visualizaciones guardadas en {fig_dir}")

def generar_tabla_kpis(escenarios):
    """
    Genera una tabla resumen de KPIs por escenario.
    """
    kpis_data = []
    
    for nombre, escenario in escenarios.items():
        sim = escenario['simulacion']
        opt = escenario['optimizacion']
        
        kpis_data.append({
            'Escenario': nombre,
            'Tiempo Cierre (min)': f"{sim['mean_cmax']:.1f}",
            'Tiempo Cierre (horas)': f"{sim['mean_cmax']/60:.1f}",
            'Prob. Horas Extra (%)': f"{sim['prob_extra']*100:.1f}",
            'Prob. >9 horas (%)': f"{sim['prob_extra_significativa']*100:.1f}",
            'Balanceo (std min)': f"{sim['balanceo_std']:.1f}",
            'Std Salida (min)': f"{sim['std_salida_entre_operarios']:.1f}",
            'Makespan (min)': f"{opt['makespan']:.1f}",
            'Eficiencia (%)': f"{(sim['mean_cmax']/(opt['n_rutas']*sim['mae_modelo']*2))*100:.0f}"
        })
    
    df_kpis = pd.DataFrame(kpis_data)
    return df_kpis

def generar_recomendaciones(escenarios, df_pool):
    """
    Genera recomendaciones operativas basadas en los resultados.
    """
    
    print("\n")
    print("RECOMENDACIONES OPERATIVAS")

    # Extraer resultados clave
    escenario_A = escenarios['A - Línea Base (3 ops)']['simulacion']
    escenario_B = escenarios['B - Día Pico (3 ops)']['simulacion']
    escenario_C = escenarios['C - Ausentismo (2 ops)']['simulacion']
    
    # Análisis de seguridad
    print("\n1. ANÁLISIS DE SEGURIDAD OPERATIVA")
    print("-" * 40)
    
    # Escenario A
    prob_A = escenario_A['prob_extra'] * 100
    if prob_A < 5:
        print(f"Escenario A (3 operarios, base): Riesgo de horas extra = {prob_A:.1f}%")
        print("   → Operación SEGURA. El sistema actual con 3 operarios tiene una probabilidad")
        print("     muy baja de generar horas extra en condiciones normales.")
    elif prob_A < 20:
        print(f"Escenario A (3 operarios, base): Riesgo de horas extra = {prob_A:.1f}%")
        print("   → Operación MARGINAL. Se recomienda monitoreo activo.")
    else:
        print(f"Escenario A (3 operarios, base): Riesgo de horas extra = {prob_A:.1f}%")
        print("   → Operación INSEGURA. Se requieren medidas correctivas.")
    
    # Escenario B (pico)
    prob_B = escenario_B['prob_extra'] * 100
    print(f"\nEscenario B (3 operarios, +20%): Riesgo de horas extra = {prob_B:.1f}%")
    if prob_B > 50:
        print("   → ALTA PROBABILIDAD de horas extra en días pico.")
        print("   → Recomendación: Aumentar plantilla a 4 operarios en días de alta demanda.")
    else:
        print("   → Riesgo moderado. Considerar estrategias de mitigación.")
    
    # Escenario C (ausentismo)
    prob_C = escenario_C['prob_extra'] * 100
    print(f"\nEscenario C (2 operarios, base): Riesgo de horas extra = {prob_C:.1f}%")
    if prob_C > 80:
        print("   → CRÍTICO: Falta de 1 operario genera horas extra casi garantizadas.")
        print("   → Recomendación: Implementar plan de contingencia con operarios de refuerzo.")
    
    # Análisis de capacidad
    print("\n2. ANÁLISIS DE CAPACIDAD")
    print("-" * 40)
    
    # Capacidad efectiva
    capacidad_actual = 3 * 480  # 3 operarios * 8 horas
    demanda_promedio = df_pool['tiempo_estimado'].sum()
    demanda_pico = demanda_promedio * 1.2
    
    print(f"Capacidad instalada: {capacidad_actual} minutos ({capacidad_actual/60:.1f} horas)")
    print(f"Demanda promedio diaria: {demanda_promedio:.0f} minutos ({demanda_promedio/60:.1f} horas)")
    print(f"Demanda pico (+20%): {demanda_pico:.0f} minutos ({demanda_pico/60:.1f} horas)")
    
    utilizacion_promedio = (demanda_promedio / capacidad_actual) * 100
    utilizacion_pico = (demanda_pico / capacidad_actual) * 100
    
    print(f"\nUtilización promedio: {utilizacion_promedio:.1f}%")
    print(f"Utilización en días pico: {utilizacion_pico:.1f}%")
    
    if utilizacion_pico > 90:
        print("La utilización en días pico es >90%, lo que indica escasa holgura.")
        print("   → Recomendación: Evaluar aumento de capacidad o ajuste de turnos.")
    
    # Recomendaciones de negocio
    print("\n3. RECOMENDACIONES ESTRATÉGICAS")
    print("-" * 40)
    
    print("\nRECOMENDACIÓN 1: Plantilla Base")
    print("   Mantener 3 operarios para operación normal. El sistema muestra:")
    print(f"   - 95% de probabilidad de cerrar en {escenario_A['mean_cmax']:.0f} minutos")
    print(f"   - Balanceo de cargas con desviación de {escenario_A['balanceo_std']:.1f} minutos")
    
    print("\nRECOMENDACIÓN 2: Gestión de Días Pico")
    print("   Implementar un sistema de alerta temprana cuando:")
    print("   - El pool de rutas supere las 65 rutas")
    print("   - El tiempo total estimado supere los 400 minutos")
    print("   - Activar operario de refuerzo (4to operario) cuando sea necesario")
    
    print("\nRECOMENDACIÓN 3: Plan de Contingencia por Ausentismo")
    print("   Desarrollar un plan de respaldo que incluya:")
    print("   - Capacitación cruzada de personal de otras áreas")
    print("   - Acuerdos con agencias de personal temporal")
    print("   - Sistema de priorización de rutas críticas")
    
    print("\nRECOMENDACIÓN 4: Mejora Continua")
    print("   - Monitorear diariamente el balanceo de cargas")
    print("   - Actualizar el modelo predictivo mensualmente con nuevos datos")
    print("   - Implementar dashboard de seguimiento en tiempo real")
    
    print("\n" + "="*80)
    print("ANÁLISIS COMPLETADO")
    print("="*80)

print("Funciones de visualización definidas")

Funciones de visualización definidas


## 6. Pipeline Principal

El pipeline principal ejecuta el flujo completo de optimización y simulación para tres escenarios operativos.

### Flujo del Pipeline

| Paso | Actividad | Descripción |
|------|-----------|-------------|
| 1 | **Carga de datos** | Cargar pool diario de rutas y predecir tiempos |
| 2 | **Optimización** | Asignar rutas a operarios con PuLP (3 escenarios) |
| 3 | **Simulación** | Ejecutar Monte Carlo para cada escenario |
| 4 | **Consolidación** | Agrupar resultados en estructura unificada |
| 5 | **Persistencia** | Guardar asignación en CSV |

### Escenarios del Pipeline

| Escenario | Operarios | Factor Demanda | Propósito |
|-----------|-----------|----------------|-----------|
| **A - Línea Base** | 3 | 1.0 | Operación estándar |
| **B - Día Pico** | 3 | 1.2 | Evaluar incremento de demanda |
| **C - Ausentismo** | 2 | 1.0 | Evaluar falta de personal |

### Parámetros Configurables

| Parámetro | Descripción | Valor por Defecto |
|-----------|-------------|-------------------|
| `n_rutas` | Número de rutas en el pool | 65 |
| `n_operarios_base` | Operarios para escenario base | 3 |
| `n_simulaciones` | Iteraciones Monte Carlo | 1000 |
| `seed` | Semilla para reproducibilidad | 42 |

### Salidas del Pipeline

| Archivo | Descripción |
|---------|-------------|
| `asignacion_diaria.csv` | Asignación de rutas a operarios (escenario base) |
| `pool_diario.csv` | Pool de rutas con tiempos estimados |
| `balanceo_cargas.png` | Gráfico de balanceo |
| `kde_escenarios.png` | Curvas de densidad |
| `kpis_comparativos.png` | KPIs normalizados |
| `kpis_escenarios.csv` | Tabla de KPIs |

In [6]:
def ejecutar_pipeline_completo(n_rutas=65, n_operarios_base=3, n_simulaciones=1000, seed=42):
    """
    Ejecuta el pipeline completo: carga, optimización y simulación.
    """
    print("="*80)
    print("INICIANDO PIPELINE")
    print("="*80)
    
    # 1. Cargar datos y predecir tiempos
    print("\nPASO 1: Carga de datos y predicciones")
    loader = DataLoaderExacto()
    df_pool = loader.cargar_pool_diario(n_rutas=n_rutas, seed=seed)
    
    # 2. Optimización para diferentes escenarios
    print("\nPASO 2: Optimización de asignación")
    
    # Escenario A: Línea Base (3 operarios)
    print("\n   Escenario A: 3 operarios - Demanda estándar")
    resultado_A = optimizar_jornada_diaria(df_pool, n_operarios=3, verbose=False, tiempo_limite=30)
    
    # Escenario B: Día Pico (3 operarios, +20% volumen)
    print("\n   Escenario B: 3 operarios - Día pico (+20%)")
    resultado_B = optimizar_jornada_diaria(df_pool, n_operarios=3, verbose=False, tiempo_limite=30)
    
    # Escenario C: Ausentismo (2 operarios)
    print("\n   Escenario C: 2 operarios - Ausentismo")
    resultado_C = optimizar_jornada_diaria(df_pool, n_operarios=2, verbose=False, tiempo_limite=30)
    
    # 3. Simulación Monte Carlo
    print("\nPASO 3: Simulación Monte Carlo")
    print(f"   {n_simulaciones} iteraciones por escenario...")
    
    # Simulador para Escenario A
    sim_A = SimuladorBodega(resultado_A['df_asignacion'], seed=seed)
    resultados_A = sim_A.simular_jornada(n_iteraciones=n_simulaciones, factor_escala=1.0)
    
    # Simulador para Escenario B
    sim_B = SimuladorBodega(resultado_B['df_asignacion'], seed=seed+100)
    resultados_B = sim_B.simular_jornada(n_iteraciones=n_simulaciones, factor_escala=1.2)
    
    # Simulador para Escenario C
    sim_C = SimuladorBodega(resultado_C['df_asignacion'], seed=seed+200)
    resultados_C = sim_C.simular_jornada(n_iteraciones=n_simulaciones, factor_escala=1.0)
    
    # 4. Consolidar resultados
    escenarios = {
        'A - Línea Base (3 ops)': {
            'optimizacion': resultado_A,
            'simulacion': resultados_A,
            'color': 'blue',
            'label': '3 Operarios - Base'
        },
        'B - Día Pico (3 ops)': {
            'optimizacion': resultado_B,
            'simulacion': resultados_B,
            'color': 'orange',
            'label': '3 Operarios - Pico +20%'
        },
        'C - Ausentismo (2 ops)': {
            'optimizacion': resultado_C,
            'simulacion': resultados_C,
            'color': 'red',
            'label': '2 Operarios - Ausentismo'
        }
    }
    
    # 5. Guardar asignaciones
    asignacion_path = os.path.join(REPORTS_PATH, 'asignacion_diaria.csv')
    resultado_A['df_asignacion'].to_csv(asignacion_path, index=False)
    print(f"\nAsignación guardada en: {asignacion_path}")
    
    print("\nPIPELINE COMPLETADO EXITOSAMENTE")
    
    return escenarios, df_pool

print("Pipeline principal definido")

Pipeline principal definido


## 7. Simulador Interactivo

Herramienta para planificar jornadas con rutas personalizadas y calcular operarios mínimos para culminar con la operación.

### Funcionalidades Principales

| # | Funcionalidad | Descripción | Método |
|---|---------------|-------------|--------|
| 1 | **Rutas personalizadas** | Seleccionar IDs de ruta específicos para simular | `seleccionar_rutas_manual(lista_rutas=[...])` |
| 2 | **Rutas aleatorias** | Generar pool aleatorio de N rutas | `seleccionar_rutas_manual(n_rutas=N)` |
| 3 | **Simulación con N operarios** | Evaluar viabilidad con número específico de operarios | `simular_con_operarios(n_operarios=N)` |
| 4 | **Cálculo de operarios mínimos** | Encontrar el mínimo necesario para jornada de 8h | `calcular_operarios_minimos()` |
| 5 | **Escenarios de demanda** | Ajustar factor de demanda (0.8 - 1.5) | `simular_con_operarios(factor_demanda=F)` |


In [7]:
# Simular jornada interactiva
class SimuladorOperativo:
    """
    Simulador interactivo para planificación de jornadas.
    Permite ingresar rutas personalizadas y calcular requisitos de personal.
    """
    
    def __init__(self):
        """Inicializa el simulador con datos reales"""
        self.loader = DataLoaderExacto()
        self.df_historico = None
        self.df_pool = None
        self._cargar_datos()
        
    def _cargar_datos(self):
        """Carga los datos históricos y prepara el pool diario"""
        try:
            # Intentar cargar datos reales
            self.df_historico = pd.read_csv(DATA_PATH)
            print(f"Datos históricos cargados: {len(self.df_historico)} registros")
        except:
            print("Generando datos sintéticos...")
            self.df_historico = self.loader._generar_datos_sinteticos(540)
    
    def seleccionar_rutas_manual(self, lista_rutas=None, n_rutas=None, seed=42):
        """
        Selecciona rutas para la simulación.
        
        Args:
            lista_rutas: Lista de IDs de ruta específicos [120, 304, 213, ...]
            n_rutas: Número de rutas aleatorias a seleccionar (si no se da lista)
            seed: Semilla para reproducibilidad
        
        Returns:
            DataFrame con las rutas seleccionadas y sus predicciones
        """
        np.random.seed(seed)
        
        if lista_rutas is not None:
            # Usar rutas específicas proporcionadas por el usuario
            print(f"Usando {len(lista_rutas)} rutas específicas")
            
            # Filtrar rutas existentes en el histórico
            ids_disponibles = set(self.df_historico['id_ruta'].astype(str).values)
            rutas_validas = [str(r) for r in lista_rutas if str(r) in ids_disponibles]
            
            if len(rutas_validas) == 0:
                print("Ninguna ruta válida encontrada. Usando rutas aleatorias...")
                rutas_validas = np.random.choice(list(ids_disponibles), size=min(20, len(ids_disponibles)), replace=False)
            
            # Seleccionar registros de esas rutas (uno por ruta)
            df_seleccion = self.df_historico[self.df_historico['id_ruta'].astype(str).isin(rutas_validas)]
            
            # Si hay múltiples registros por ruta, tomar el más reciente
            if 'fecha' in df_seleccion.columns:
                df_seleccion = df_seleccion.sort_values('fecha', ascending=False).groupby('id_ruta').first().reset_index()
            else:
                df_seleccion = df_seleccion.groupby('id_ruta').first().reset_index()
            
            df_pool = df_seleccion
            
        elif n_rutas is not None:
            # Seleccionar N rutas aleatorias
            print(f"Seleccionando {n_rutas} rutas aleatorias")
            
            # Obtener rutas únicas
            rutas_unicas = self.df_historico['id_ruta'].unique()
            n_seleccion = min(n_rutas, len(rutas_unicas))
            rutas_seleccionadas = np.random.choice(rutas_unicas, size=n_seleccion, replace=False)
            
            # Seleccionar un registro por ruta (el más reciente)
            df_seleccion = self.df_historico[self.df_historico['id_ruta'].isin(rutas_seleccionadas)]
            if 'fecha' in df_seleccion.columns:
                df_seleccion = df_seleccion.sort_values('fecha', ascending=False).groupby('id_ruta').first().reset_index()
            else:
                df_seleccion = df_seleccion.groupby('id_ruta').first().reset_index()
            
            df_pool = df_seleccion
            
        else:
            # Default: 65 rutas aleatorias
            print("Usando pool estándar de 65 rutas")
            df_pool = self.loader.cargar_pool_diario(n_rutas=65, seed=seed)
            self.df_pool = df_pool
            return df_pool
        
        # Generar IDs de ruta consistentes
        df_pool['id_ruta'] = df_pool['id_ruta'].astype(str)
        
        # Reconstruir features y predecir
        try:
            df_pool = self.loader._reconstruir_features(df_pool)
            df_pool = self.loader._predecir_tiempos(df_pool)
        except:
            print("Error al reconstruir features. Usando datos existentes...")
            # Si no se pueden reconstruir, usar tiempos existentes o estimar
            if 'tiempo_estimado' not in df_pool.columns:
                df_pool['tiempo_estimado'] = np.random.normal(16, 8, len(df_pool))
                df_pool['tiempo_estimado'] = np.maximum(df_pool['tiempo_estimado'], 3)
        
        print(f"{len(df_pool)} rutas preparadas para simulación")
        print(f"   Tiempo total estimado: {df_pool['tiempo_estimado'].sum():.1f} min ({df_pool['tiempo_estimado'].sum()/60:.1f} horas)")
        
        self.df_pool = df_pool
        return df_pool
    
    def simular_con_operarios(self, n_operarios, factor_demanda=1.0, n_iteraciones=1000):
        """
        Simula la jornada con un número específico de operarios.
        
        Args:
            n_operarios: Número de operarios disponibles
            factor_demanda: Factor de escala de la demanda (1.0 = normal)
            n_iteraciones: Número de iteraciones Monte Carlo
        
        Returns:
            dict con resultados detallados
        """
        if self.df_pool is None:
            print("No hay rutas seleccionadas. Usando pool estándar...")
            self.seleccionar_rutas_manual(n_rutas=65)
        
        print("\n" + "="*70)
        print(f"SIMULANDO CON {n_operarios} OPERARIOS")
        print("="*70)
        print(f"Rutas: {len(self.df_pool)}")
        print(f"Demanda: {factor_demanda:.0%} de la normal")
        print(f"Tiempo total base: {self.df_pool['tiempo_estimado'].sum():.1f} min")
        print("="*70)
        
        # Optimizar asignación
        resultado = optimizar_jornada_diaria(
            self.df_pool,
            n_operarios=n_operarios,
            verbose=False
        )
        
        # Simular con Monte Carlo
        simulador = SimuladorBodega(resultado['df_asignacion'], seed=42)
        resultados_sim = simulador.simular_jornada(
            n_iteraciones=n_iteraciones,
            factor_escala=factor_demanda
        )
        
        # Calcular métricas clave
        makespan_promedio = resultados_sim['mean_cmax']
        prob_horas_extra = resultados_sim['prob_extra']
        horas_extra_promedio = max(0, makespan_promedio - 480) / 60
        balanceo = resultados_sim['balanceo_std']
        
        # Determinar viabilidad
        viable = makespan_promedio <= 480
        nivel_riesgo = self._evaluar_riesgo(prob_horas_extra)
        
        # Resultados
        resultados = {
            'n_operarios': n_operarios,
            'factor_demanda': factor_demanda,
            'n_rutas': len(self.df_pool),
            'tiempo_total': self.df_pool['tiempo_estimado'].sum(),
            'makespan_promedio': makespan_promedio,
            'makespan_horas': makespan_promedio / 60,
            'prob_horas_extra': prob_horas_extra * 100,
            'horas_extra_promedio': horas_extra_promedio,
            'balanceo': balanceo,
            'viable': viable,
            'nivel_riesgo': nivel_riesgo,
            'cargas_operarios': resultado['cargas_operarios'],
            'desviacion_cargas': resultado['desviacion_cargas']
        }
        
        self._mostrar_resultados(resultados)
        
        return resultados
    
    def _evaluar_riesgo(self, prob_horas_extra):
        """Evalúa el nivel de riesgo según la probabilidad de horas extra"""
        if prob_horas_extra == 0:
            return "SEGURO"
        elif prob_horas_extra < 0.20:
            return "BAJO RIESGO"
        elif prob_horas_extra < 0.50:
            return "RIESGO MODERADO"
        elif prob_horas_extra < 0.80:
            return "ALTO RIESGO"
        else:
            return "RIESGO CRÍTICO"
    
    def _mostrar_resultados(self, resultados):
        """Muestra los resultados de la simulación"""
        print("\n" + "="*70)
        print("RESULTADOS DE LA SIMULACIÓN")
        print("="*70)
        
        # Métricas principales
        print(f"\nMAKESPAN:")
        print(f"   Promedio: {resultados['makespan_promedio']:.1f} min ({resultados['makespan_horas']:.1f} horas)")
        
        # Estado de la jornada
        if resultados['viable']:
            holgura = 480 - resultados['makespan_promedio']
            print(f"JORNADA VIABLE - {holgura:.1f} min ({holgura/60:.1f} horas) de holgura")
        else:
            exceso = resultados['makespan_promedio'] - 480
            print(f"JORNADA NO VIABLE - {exceso:.1f} min ({exceso/60:.1f} horas) EXCEDEN la jornada")
        
        # Horas extra
        print(f"\nHORAS EXTRA:")
        print(f"   Probabilidad: {resultados['prob_horas_extra']:.1f}%")
        print(f"   Promedio requerido: {resultados['horas_extra_promedio']:.1f} horas")
        print(f"   Nivel de riesgo: {resultados['nivel_riesgo']}")
        
        # Balanceo
        print(f"\nBALANCEO DE CARGAS:")
        print(f"   Cargas por operario: {[f'{c:.1f} min ({c/60:.1f}h)' for c in resultados['cargas_operarios']]}")
        print(f"   Desviación estándar: {resultados['balanceo']:.1f} min")
        if np.mean(resultados['cargas_operarios']) > 0:
            cv = (resultados['balanceo'] / np.mean(resultados['cargas_operarios']) * 100)
            print(f"   Coeficiente de variación: {cv:.1f}%")
        
        # Resumen
        print("\n" + "="*70)
        if resultados['viable']:
            print("RECOMENDACIÓN: La operación es VIABLE con los recursos disponibles")
        else:
            print(f"RECOMENDACIÓN: Se necesitan {resultados['horas_extra_promedio']:.1f} horas extra")
            print(f"   → Considere: Aumentar a {resultados['n_operarios'] + 1} operarios o reducir rutas")
        print("="*70)
    
    def calcular_operarios_minimos(self, factor_demanda=1.0, max_operarios=10):
        """
        Calcula el número mínimo de operarios necesarios para la jornada de 8 horas.
        
        Args:
            factor_demanda: Factor de escala de la demanda
            max_operarios: Máximo número de operarios a evaluar
        
        Returns:
            dict con el número mínimo de operarios y detalles
        """
        if self.df_pool is None:
            print("No hay rutas seleccionadas. Usando pool estándar...")
            self.seleccionar_rutas_manual(n_rutas=65)
        
        print("\n" + "="*70)
        print("CALCULANDO MÍNIMO DE OPERARIOS REQUERIDOS")
        print("="*70)
        print(f"Rutas: {len(self.df_pool)}")
        print(f"Demanda: {factor_demanda:.0%} de la normal")
        print(f"Tiempo total: {self.df_pool['tiempo_estimado'].sum():.1f} min")
        print(f"Meta: Jornada de 480 min (8 horas)")
        print("="*70)
        
        resultados = []
        
        for n_ops in range(1, max_operarios + 1):
            # Simular con este número de operarios
            resultado = self.simular_con_operarios(n_ops, factor_demanda, n_iteraciones=200)
            
            resultados.append({
                'operarios': n_ops,
                'makespan_min': resultado['makespan_promedio'],
                'makespan_horas': resultado['makespan_horas'],
                'viable': resultado['viable'],
                'prob_horas_extra': resultado['prob_horas_extra'],
                'horas_extra': resultado['horas_extra_promedio'],
                'balanceo': resultado['balanceo']
            })
        
        # Encontrar el mínimo viable
        df_resultados = pd.DataFrame(resultados)
        minimo_viable = df_resultados[df_resultados['viable']].iloc[0] if len(df_resultados[df_resultados['viable']]) > 0 else None
        
        print("\n" + "="*70)
        print("RESULTADOS POR NÚMERO DE OPERARIOS")
        print("="*70)
        print(df_resultados.round(1).to_string(index=False))
        
        # Visualización
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Gráfico 1: Makespan vs Operarios
        ax1 = axes[0]
        ax1.plot(df_resultados['operarios'], df_resultados['makespan_horas'], 
                'bo-', markersize=8, linewidth=2)
        ax1.axhline(y=8, color='red', linestyle='--', linewidth=2, label='Límite 8 horas')
        
        # Marcar el mínimo viable
        if minimo_viable is not None:
            ax1.plot(minimo_viable['operarios'], minimo_viable['makespan_horas'], 
                    'g*', markersize=15, label=f'Mínimo viable: {int(minimo_viable["operarios"])} ops')
        
        ax1.set_xlabel('Número de Operarios')
        ax1.set_ylabel('Makespan (horas)')
        ax1.set_title('Tiempo de Cierre vs Operarios Disponibles')
        ax1.grid(True, alpha=0.3)
        ax1.legend()
        
        # Agregar etiquetas
        for _, row in df_resultados.iterrows():
            ax1.annotate(f'{row["makespan_horas"]:.1f}h', 
                        (row['operarios'], row['makespan_horas']),
                        textcoords="offset points", xytext=(0,10), ha='center')
        
        # Gráfico 2: Probabilidad de horas extra
        ax2 = axes[1]
        colors = ['green' if p == 0 else 'orange' if p < 50 else 'red' 
                  for p in df_resultados['prob_horas_extra']]
        ax2.bar(df_resultados['operarios'], df_resultados['prob_horas_extra'], 
                color=colors, edgecolor='black')
        ax2.axhline(y=50, color='orange', linestyle='--', alpha=0.5, label='Riesgo 50%')
        ax2.set_xlabel('Número de Operarios')
        ax2.set_ylabel('Probabilidad de Horas Extra (%)')
        ax2.set_title('Riesgo de Horas Extra vs Operarios')
        ax2.grid(True, alpha=0.3)
        ax2.legend()
        
        # Agregar valores
        for _, row in df_resultados.iterrows():
            ax2.text(row['operarios'], row['prob_horas_extra'] + 2,
                    f'{row["prob_horas_extra"]:.0f}%',
                    ha='center', fontweight='bold')
        
        plt.tight_layout()
        plt.savefig(os.path.join(REPORTS_PATH, 'operarios_minimos.png'), dpi=150)
        plt.show()
        
        # Recomendación final
        print("\n" + "="*70)
        print("RECOMENDACIÓN FINAL")
        print("="*70)
        
        if minimo_viable is not None:
            print(f"\nSe requieren MÍNIMO {int(minimo_viable['operarios'])} operarios")
            print(f"   Makespan: {minimo_viable['makespan_horas']:.1f} horas")
            print(f"   Holgura: {8 - minimo_viable['makespan_horas']:.1f} horas")
            print(f"   Riesgo de horas extra: {minimo_viable['prob_horas_extra']:.1f}%")
            
            # Recomendación adicional
            if minimo_viable['operarios'] == 1:
                print("\n1 operario es el mínimo absoluto - NO RECOMENDADO")
                print("   → Considere al menos 2 operarios para tener holgura")
            elif minimo_viable['prob_horas_extra'] > 20:
                print(f"\nRiesgo de horas extra del {minimo_viable['prob_horas_extra']:.1f}%")
                print(f"   → Considere agregar 1 operario más para reducir riesgo")
            else:
                print(f"\nConfiguración segura - {int(minimo_viable['operarios'])} operarios es suficiente")
        else:
            print(f"\nCon {max_operarios} operarios NO es suficiente")
            print(f"   → Se necesitan más de {max_operarios} operarios para esta demanda")
        
        print("="*70)
        
        return {
            'minimo_viable': minimo_viable.to_dict() if minimo_viable is not None else None,
            'todos_resultados': df_resultados.to_dict('records')
        }

print("SimuladorOperativo definido correctamente")

SimuladorOperativo definido correctamente


## 8. Ejemplos de Uso del Simulador Operativo

Esta sección demuestra cómo utilizar el `SimuladorOperativo` en diferentes escenarios operativos, desde planificación diaria hasta análisis de contingencia.

### Tipos de Ejemplos

| # | Ejemplo | Descripción | Función |
|---|---------|-------------|---------|
| 1 | **Rutas específicas** | Simular con IDs de ruta concretos | `sim.seleccionar_rutas_manual(lista_rutas=[...])` |
| 2 | **Rutas aleatorias** | Simular con N rutas aleatorias | `sim.seleccionar_rutas_manual(n_rutas=N)` |
| 3 | **Cálculo de operarios** | Encontrar mínimo de operarios | `sim.calcular_operarios_minimos()` |
| 4 | **Día pico** | Evaluar demanda incrementada (+20%) | `sim.simular_con_operarios(3, factor_demanda=1.2)` |
| 5 | **Función rápida** | Todo-en-uno para uso diario | `planificar_jornada(rutas=[...], n_operarios=3)` |

In [8]:
def planificar_jornada(rutas=None, n_operarios=3, n_rutas=None, factor_demanda=1.0):
    """
    Función rápida para planificar una jornada.
    
    Args:
        rutas: Lista de IDs de ruta (opcional)
        n_operarios: Número de operarios disponibles
        n_rutas: Número de rutas aleatorias (si no se da lista)
        factor_demanda: Factor de escala de la demanda
    
    Returns:
        tuple: (resultado, minimo)
            - resultado: Dict con resultados de la simulación
            - minimo: Dict con mínimo de operarios requeridos
    
    Ejemplos:
        # Con rutas específicas
        resultado, minimo = planificar_jornada(
            rutas=[7603, 7606, 7646, 8002],
            n_operarios=3
        )
        
        # Con rutas aleatorias
        resultado, minimo = planificar_jornada(
            n_rutas=50,
            n_operarios=2
        )
        
        # En día pico
        resultado, minimo = planificar_jornada(
            n_rutas=65,
            n_operarios=3,
            factor_demanda=1.2
        )
    """
    print("\n" + "="*70)
    print("PLANIFICADOR DE JORNADA")
    print("="*70)
    
    # Crear simulador
    sim = SimuladorOperativo()
    
    # Seleccionar rutas
    if rutas is not None:
        df_pool = sim.seleccionar_rutas_manual(lista_rutas=rutas)
    elif n_rutas is not None:
        df_pool = sim.seleccionar_rutas_manual(n_rutas=n_rutas)
    else:
        df_pool = sim.seleccionar_rutas_manual(n_rutas=65)
    
    # Simular
    print("\n" + "="*70)
    print("EJECUTANDO SIMULACIÓN")
    print("="*70)
    resultado = sim.simular_con_operarios(n_operarios, factor_demanda)
    
    # Calcular mínimo de operarios
    print("\n" + "="*70)
    print("CALCULANDO MÍNIMO DE OPERARIOS")
    print("="*70)
    minimo = sim.calcular_operarios_minimos(factor_demanda, max_operarios=8)
    
    print("\n" + "="*70)
    print("PLANIFICACIÓN COMPLETADA")
    print("="*70)
    
    return resultado, minimo

# ============================================================
# EJEMPLOS DE USO - EJECUTAR ESTA CELDA PARA VER EJEMPLOS
# ============================================================

print("\n" + "="*80)
print("EJEMPLO 1: Simular con rutas específicas")
print("="*80)

# Crear simulador
sim = SimuladorOperativo()

# Usar rutas específicas
rutas_personalizadas = [7603, 7606, 7646, 8002, 209, 253, 256, 260, 213, 304]
df_pool = sim.seleccionar_rutas_manual(lista_rutas=rutas_personalizadas)

# Simular con 3 operarios
resultado = sim.simular_con_operarios(n_operarios=3)

print("\n" + "="*80)
print("EJEMPLO 2: Calcular mínimo de operarios")
print("="*80)

# Calcular mínimo de operarios necesarios
minimo = sim.calcular_operarios_minimos()

print("\n" + "="*80)
print("EJEMPLO 3: Probar diferentes números de operarios")
print("="*80)

# Probar diferentes números de operarios
for n_ops in [2, 3, 4]:
    print(f"\n--- Con {n_ops} operarios ---")
    sim.simular_con_operarios(n_ops, factor_demanda=1.0)

print("\n" + "="*80)
print("EJEMPLO 4: Simular escenario de día pico (+20%)")
print("="*80)

# Simular día pico con 3 operarios
resultado_pico = sim.simular_con_operarios(n_operarios=3, factor_demanda=1.2)

print("\n" + "="*80)
print("EJEMPLO 5: Usar función rápida planificar_jornada()")
print("="*80)

# Usar la función rápida
resultado_rapido, minimo_rapido = planificar_jornada(
    rutas=[7603, 7606, 7646, 8002, 209, 253],
    n_operarios=3
)

print("\n" + "="*80)
print("EJEMPLO 6: Simular con 50 rutas aleatorias y 2 operarios")
print("="*80)

# 50 rutas aleatorias con 2 operarios
resultado_50, minimo_50 = planificar_jornada(
    n_rutas=50,
    n_operarios=2
)

print("\n" + "="*80)
print("TODOS LOS EJEMPLOS COMPLETADOS EXITOSAMENTE")
print("="*80)


EJEMPLO 1: Simular con rutas específicas
Verificando archivos de modelos...
scaler.pkl encontrado
random_forest_model.pkl encontrado
metadatos.pkl encontrado
Scaler cargado correctamente
Modelo Random Forest cargado correctamente
Metadatos cargados correctamente

Artefactos cargados exitosamente desde /home/migue07/Documents/Proyectos/optimizacion_despachos/models/
   - Modelo: RandomForestRegressor
   - Features esperadas: 8 features
   - MAE del modelo real: 3.554192374250871 min
   - R² del modelo real: 0.7637844537840316
Datos históricos cargados: 560 registros
Usando 10 rutas específicas
Error al escalar features: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- cant_productos
- dia_semana
- es_jueves
- es_lunes_o_viernes
- es_ruta_flash
- ...
Feature names seen at fit time, yet now missing:
- cant_productos_promedio
- valor_ruta_promedio

   Usando datos sin escalar...
10 rutas preparadas para simulación
   Tiempo total estima

KeyError: 10

## 9. Ejecución Principal del Pipeline

Esta sección ejecuta el pipeline completo con los parámetros configurados, generando resultados consolidados y visualizaciones.

### Flujo de Ejecución

| Paso | Descripción |
|------|-------------|
| 1 | **Configurar parámetros** (rutas, operarios, simulaciones) |
| 2 | **Ejecutar pipeline** completo (carga, optimización, simulación) |
| 3 | **Generar visualizaciones** (balanceo, KDE, KPIs) |
| 4 | **Calcular KPIs** y mostrar tabla resumen |
| 5 | **Guardar resultados** (CSV, gráficos) |
| 6 | **Generar recomendaciones** operativas |

In [ ]:

# Parámetros configurables
SEED = 42
N_RUTAS = 65
N_OPERARIOS = 3
N_SIMULACIONES = 100  # Reducido de 1000 a 100 para pruebas rápidas

# Mostrar configuración
print("\nCONFIGURACIÓN:")
print(f"   • Semilla (SEED): {SEED}")
print(f"   • Rutas (N_RUTAS): {N_RUTAS}")
print(f"   • Operarios (N_OPERARIOS): {N_OPERARIOS}")
print(f"   • Simulaciones (N_SIMULACIONES): {N_SIMULACIONES}")
print("="*80)

# Verificar configuración
print("\nVerificando configuración...")
if N_SIMULACIONES < 50:
    print("N_SIMULACIONES es bajo (< 50). Resultados pueden no ser estadísticamente significativos.")
if N_RUTAS < 10:
    print("N_RUTAS es bajo (< 10). Considerar aumentar para mejor representación.")
if N_OPERARIOS < 1:
    print("N_OPERARIOS debe ser >= 1")
    raise ValueError("N_OPERARIOS debe ser >= 1")
print("Configuración validada")

# ============================================================
# EJECUCIÓN DEL PIPELINE
# ============================================================

try:
    # Ejecutar pipeline completo
    escenarios, df_pool = ejecutar_pipeline_completo(
        n_rutas=N_RUTAS,
        n_operarios_base=N_OPERARIOS,
        n_simulaciones=N_SIMULACIONES,
        seed=SEED
    )
except Exception as e:
    print(f"\nError en la ejecución del pipeline: {e}")
    raise

# ============================================================
# VISUALIZACIONES Y KPIS
# ============================================================

# Generar visualizaciones
print("\n" + "="*80)
print("GENERANDO VISUALIZACIONES")
print("="*80)
generar_visualizaciones(escenarios, df_pool)

# Generar tabla de KPIs
print("\n" + "="*80)
print("TABLA RESUMEN DE KPIs")
print("="*80)
df_kpis = generar_tabla_kpis(escenarios)
print(df_kpis.to_string(index=False))

# Guardar KPIs
kpi_path = os.path.join(REPORTS_PATH, 'kpis_escenarios.csv')
df_kpis.to_csv(kpi_path, index=False)
print(f"\nKPIs guardados en: {kpi_path}")

# ============================================================
# RECOMENDACIONES
# ============================================================

# Generar recomendaciones
generar_recomendaciones(escenarios, df_pool)

# ============================================================
# GUARDAR RESULTADOS ADICIONALES
# ============================================================

# Guardar datos del pool para referencia
pool_path = os.path.join(REPORTS_PATH, 'pool_diario.csv')
df_pool[['id_ruta', 'cant_productos', 'valor_ruta', 'dia_semana', 'tiempo_estimado']].to_csv(pool_path, index=False)
print(f"\nPool diario guardado en: {pool_path}")

# ============================================================
# RESUMEN FINAL
# ============================================================

print("\n" + "="*80)
print("RESUMEN FINAL DE RESULTADOS")
print("="*80)

for nombre, escenario in escenarios.items():
    sim = escenario['simulacion']
    opt = escenario['optimizacion']
    status = "VIABLE" if sim['mean_cmax'] <= 480 else "NO VIABLE"
    print(f"\n{nombre}:")
    print(f"   • Makespan: {sim['mean_cmax']:.1f} min ({sim['mean_cmax']/60:.1f} horas) {status}")
    print(f"   • Prob. Horas Extra: {sim['prob_extra']*100:.1f}%")
    print(f"   • Balanceo: {sim['balanceo_std']:.1f} min")
    print(f"   • Cargas: {[f'{c:.1f} min' for c in opt['cargas_operarios']]}")

# Identificar escenario más crítico
escenarios_con_riesgo = [(nombre, sim['prob_extra']) for nombre, sim in 
                         [(n, e['simulacion']) for n, e in escenarios.items()]]
escenario_critico = max(escenarios_con_riesgo, key=lambda x: x[1])
if escenario_critico[1] > 0.5:
    print(f"\nESCENARIO CRÍTICO IDENTIFICADO:")
    print(f"   • {escenario_critico[0]} con {escenario_critico[1]*100:.1f}% probabilidad de horas extra")
    print(f"   • Recomendación: Implementar plan de contingencia inmediato")

print("\n" + "="*80)
print("PROCESO COMPLETADO EXITOSAMENTE")
print("="*80)
print("\nResultados guardados en:")
print(f"   • Visualizaciones: {REPORTS_PATH}")
print(f"   • KPIs: {kpi_path}")
print(f"   • Pool diario: {pool_path}")
print("\nSugerencias para próximas ejecuciones:")
print(f"   • Para mayor precisión: aumentar N_SIMULACIONES a 1000")
print(f"   • Para análisis de capacidad: probar diferentes N_OPERARIOS")
print(f"   • Para planificación: usar el SimuladorOperativo con rutas personalizadas")
print("="*80)

## 10. Conclusciones y Recomendaciones

La Fase 3 ha demostrado que la optimización matemática combinada con simulación de escenarios permite:

### Logros Alcanzados
- **Reducción de horas extra**: Optimización de asignación reduce significativamente el tiempo extra requerido
- **Equidad laboral**: Distribución balanceada de carga entre operarios
- **Robustez operativa**: Identificación de puntos críticos bajo escenarios de estrés
- **Simulación interactiva**: Herramienta para planificar con rutas personalizadas
 
### Metodología Aplicada
1. **Predicción:** Random Forest para estimar tiempos de preparación
2. **Optimización:** Programación Lineal Entera (PuLP) para asignación óptima
3. **Simulación:** Monte Carlo (SimPy) para evaluación de escenarios
4. **Planificación:** Simulador interactivo para toma de decisiones